In [1]:
import pandas as pd
import numpy as np
import sqlite3
import os

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

DATA_PATH = '../data/'
DB_PATH   = '../data/olist.db'

print('Libraries loaded ✓')

Libraries loaded ✓


In [2]:
tables = ['orders', 'order_items', 'order_payments',
          'order_reviews', 'customers', 'sellers',
          'products', 'geolocation', 'product_category_translation']

dfs = {}
for name in tables:
    dfs[name] = pd.read_csv(f'{DATA_PATH}cleaned_{name}.csv')
    print(f'  ✓  {name:<35} {dfs[name].shape[0]:>8,} rows')

print('\nAll tables loaded ✓')

  ✓  orders                                99,441 rows
  ✓  order_items                          112,650 rows
  ✓  order_payments                       103,886 rows
  ✓  order_reviews                         99,224 rows
  ✓  customers                             99,441 rows
  ✓  sellers                                3,095 rows
  ✓  products                              32,951 rows
  ✓  geolocation                           27,912 rows
  ✓  product_category_translation              71 rows

All tables loaded ✓


In [3]:
conn = sqlite3.connect(DB_PATH)

for name, df in dfs.items():
    df.to_sql(name, conn, if_exists='replace', index=False)
    print(f'  ✓  Table "{name}" written to database')

print(f'\nDatabase created at: {DB_PATH}')

  ✓  Table "orders" written to database
  ✓  Table "order_items" written to database
  ✓  Table "order_payments" written to database
  ✓  Table "order_reviews" written to database
  ✓  Table "customers" written to database
  ✓  Table "sellers" written to database
  ✓  Table "products" written to database
  ✓  Table "geolocation" written to database
  ✓  Table "product_category_translation" written to database

Database created at: ../data/olist.db


In [4]:
query = "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;"
tables_in_db = pd.read_sql(query, conn)
print('Tables in olist.db:')
print(tables_in_db)

Tables in olist.db:
                           name
0                     customers
1                   geolocation
2                   order_items
3                order_payments
4                 order_reviews
5                        orders
6  product_category_translation
7                      products
8                       sellers


In [5]:
# This function lets us run any SQL query and see results cleanly
def run_query(sql, title=''):
    if title:
        print(f'\n{"="*60}')
        print(f'  {title}')
        print(f'{"="*60}')
    result = pd.read_sql(sql, conn)
    display(result)
    return result

In [6]:
run_query("""
    SELECT
        COUNT(DISTINCT o.order_id)          AS total_orders,
        COUNT(DISTINCT c.customer_unique_id) AS total_customers,
        ROUND(SUM(p.payment_value), 2)       AS total_revenue,
        ROUND(AVG(p.payment_value), 2)       AS avg_order_value
    FROM orders o
    JOIN customers c        ON o.customer_id  = c.customer_id
    JOIN order_payments p   ON o.order_id     = p.order_id
    WHERE o.order_status = 'delivered'
""", "Q1: Overall Business Performance")


  Q1: Overall Business Performance


,total_orders,total_customers,total_revenue,avg_order_value
0,96477,93357,15422461.77,153.07


,total_orders,total_customers,total_revenue,avg_order_value
0,96477,93357,15422461.77,153.07


In [7]:
run_query("""
    SELECT
        SUBSTR(o.order_purchase_timestamp, 1, 7)  AS year_month,
        COUNT(DISTINCT o.order_id)                AS total_orders,
        ROUND(SUM(p.payment_value), 2)            AS monthly_revenue
    FROM orders o
    JOIN order_payments p ON o.order_id = p.order_id
    WHERE o.order_status = 'delivered'
    GROUP BY year_month
    ORDER BY year_month
""", "Q2: Monthly Revenue Trend")


  Q2: Monthly Revenue Trend


,year_month,total_orders,monthly_revenue
0,2016-10,265,46566.71
1,2016-12,1,19.62
2,2017-01,750,127545.67
3,2017-02,1653,271298.65
4,2017-03,2546,414369.39
5,2017-04,2303,390952.18
6,2017-05,3546,567066.73
7,2017-06,3135,490225.60
8,2017-07,3872,566403.93
9,2017-08,4193,646000.61


,year_month,total_orders,monthly_revenue
0,2016-10,265,46566.71
1,2016-12,1,19.62
2,2017-01,750,127545.67
3,2017-02,1653,271298.65
4,2017-03,2546,414369.39
5,2017-04,2303,390952.18
6,2017-05,3546,567066.73
7,2017-06,3135,490225.60
8,2017-07,3872,566403.93
9,2017-08,4193,646000.61


In [8]:
run_query("""
    SELECT
        t.product_category_name_english         AS category,
        COUNT(DISTINCT oi.order_id)             AS total_orders,
        ROUND(SUM(oi.price), 2)                 AS total_revenue,
        ROUND(AVG(oi.price), 2)                 AS avg_price
    FROM order_items oi
    JOIN products p     ON oi.product_id  = p.product_id
    JOIN product_category_translation t
                        ON p.product_category_name = t.product_category_name
    GROUP BY category
    ORDER BY total_revenue DESC
    LIMIT 10
""", "Q3: Top 10 Categories by Revenue")


  Q3: Top 10 Categories by Revenue


,category,total_orders,total_revenue,avg_price
0,health_beauty,8836,1258681.34,130.16
1,watches_gifts,5624,1205005.68,201.14
2,bed_bath_table,9417,1036988.68,93.30
3,sports_leisure,7720,988048.97,114.34
4,computers_accessories,6689,911954.32,116.51
5,furniture_decor,6449,729762.49,87.56
6,cool_stuff,3632,635290.85,167.36
7,housewares,5884,632248.66,90.79
8,auto,3897,592720.11,139.96
9,garden_tools,3518,485256.46,111.63


,category,total_orders,total_revenue,avg_price
0,health_beauty,8836,1258681.34,130.16
1,watches_gifts,5624,1205005.68,201.14
2,bed_bath_table,9417,1036988.68,93.30
3,sports_leisure,7720,988048.97,114.34
4,computers_accessories,6689,911954.32,116.51
5,furniture_decor,6449,729762.49,87.56
6,cool_stuff,3632,635290.85,167.36
7,housewares,5884,632248.66,90.79
8,auto,3897,592720.11,139.96
9,garden_tools,3518,485256.46,111.63


In [9]:
run_query("""
    SELECT
        COUNT(*)                                          AS total_delivered,
        ROUND(AVG(JULIANDAY(order_delivered_customer_date)
              - JULIANDAY(order_purchase_timestamp)), 1)  AS avg_delivery_days,
        ROUND(MIN(JULIANDAY(order_delivered_customer_date)
              - JULIANDAY(order_purchase_timestamp)), 1)  AS min_delivery_days,
        ROUND(MAX(JULIANDAY(order_delivered_customer_date)
              - JULIANDAY(order_purchase_timestamp)), 1)  AS max_delivery_days,
        SUM(CASE
              WHEN order_delivered_customer_date > order_estimated_delivery_date
              THEN 1 ELSE 0
            END)                                          AS late_orders,
        ROUND(
            100.0 * SUM(CASE
                WHEN order_delivered_customer_date > order_estimated_delivery_date
                THEN 1 ELSE 0
            END) / COUNT(*), 1)                           AS late_pct
    FROM orders
    WHERE order_status = 'delivered'
      AND order_delivered_customer_date IS NOT NULL
""", "Q4: Delivery Performance")


  Q4: Delivery Performance


,total_delivered,avg_delivery_days,min_delivery_days,max_delivery_days,late_orders,late_pct
0,96470,12.60,0.50,209.60,7826,8.10


,total_delivered,avg_delivery_days,min_delivery_days,max_delivery_days,late_orders,late_pct
0,96470,12.60,0.50,209.60,7826,8.10


In [10]:
run_query("""
    SELECT
        oi.seller_id,
        s.seller_city,
        s.seller_state,
        COUNT(DISTINCT oi.order_id)   AS total_orders,
        ROUND(SUM(oi.price), 2)       AS total_revenue,
        ROUND(AVG(oi.price), 2)       AS avg_item_price
    FROM order_items oi
    JOIN sellers s ON oi.seller_id = s.seller_id
    GROUP BY oi.seller_id
    ORDER BY total_revenue DESC
    LIMIT 10
""", "Q5: Top 10 Sellers by Revenue")


  Q5: Top 10 Sellers by Revenue


,seller_id,seller_city,seller_state,total_orders,total_revenue,avg_item_price
0,4869f7a5dfa277a7dca6462dcf3b52b2,guariba,SP,1132,229472.63,198.51
1,53243585a1d6dc2643021fd1853d8905,lauro de freitas,BA,358,222776.05,543.36
2,4a3ca9315b744ce9f8e9374361493884,ibitinga,SP,1806,200472.92,100.89
3,fa1c13f2614d7b5c4749cbc52fecda94,sumare,SP,585,194042.03,331.13
4,7c67e1448b00f6e969d365cea6b010ab,itaquaquecetuba,SP,982,187923.89,137.77
5,7e93a43ef30c4f03f38b393420bc753a,barueri,SP,336,176431.87,518.92
6,da8622b14eb17ae2831f4ac5b9dab84a,piracicaba,SP,1314,160236.57,103.31
7,7a67c85e85bb2ce8582c35f2203ad736,sao paulo,SP,1160,141745.53,121.05
8,1025f0e2d44d7041d6cf58b6550e0bfa,sao paulo,SP,915,138968.55,97.32
9,955fee9216a65b617aa5c0531780ce60,sao paulo,SP,1287,135171.70,90.17


,seller_id,seller_city,seller_state,total_orders,total_revenue,avg_item_price
0,4869f7a5dfa277a7dca6462dcf3b52b2,guariba,SP,1132,229472.63,198.51
1,53243585a1d6dc2643021fd1853d8905,lauro de freitas,BA,358,222776.05,543.36
2,4a3ca9315b744ce9f8e9374361493884,ibitinga,SP,1806,200472.92,100.89
3,fa1c13f2614d7b5c4749cbc52fecda94,sumare,SP,585,194042.03,331.13
4,7c67e1448b00f6e969d365cea6b010ab,itaquaquecetuba,SP,982,187923.89,137.77
5,7e93a43ef30c4f03f38b393420bc753a,barueri,SP,336,176431.87,518.92
6,da8622b14eb17ae2831f4ac5b9dab84a,piracicaba,SP,1314,160236.57,103.31
7,7a67c85e85bb2ce8582c35f2203ad736,sao paulo,SP,1160,141745.53,121.05
8,1025f0e2d44d7041d6cf58b6550e0bfa,sao paulo,SP,915,138968.55,97.32
9,955fee9216a65b617aa5c0531780ce60,sao paulo,SP,1287,135171.70,90.17


In [11]:
run_query("""
    SELECT
        review_score,
        COUNT(*)                                     AS total_reviews,
        ROUND(100.0 * COUNT(*) /
              (SELECT COUNT(*) FROM order_reviews), 1) AS percentage
    FROM order_reviews
    GROUP BY review_score
    ORDER BY review_score
""", "Q6: Review Score Distribution")


  Q6: Review Score Distribution


,review_score,total_reviews,percentage
0,1,11424,11.50
1,2,3151,3.20
2,3,8179,8.20
3,4,19142,19.30
4,5,57328,57.80


,review_score,total_reviews,percentage
0,1,11424,11.50
1,2,3151,3.20
2,3,8179,8.20
3,4,19142,19.30
4,5,57328,57.80


In [12]:
run_query("""
    SELECT
        payment_type,
        COUNT(DISTINCT order_id)          AS total_orders,
        ROUND(SUM(payment_value), 2)      AS total_revenue,
        ROUND(AVG(payment_value), 2)      AS avg_payment,
        ROUND(AVG(payment_installments), 1) AS avg_installments
    FROM order_payments
    GROUP BY payment_type
    ORDER BY total_revenue DESC
""", "Q7: Payment Method Analysis")


  Q7: Payment Method Analysis


,payment_type,total_orders,total_revenue,avg_payment,avg_installments
0,credit_card,76505,12542084.19,163.32,3.50
1,boleto,19784,2869361.27,145.03,1.00
2,voucher,3866,379436.87,65.70,1.00
3,debit_card,1528,217989.79,142.57,1.00
4,not_defined,3,0.00,0.00,1.00


,payment_type,total_orders,total_revenue,avg_payment,avg_installments
0,credit_card,76505,12542084.19,163.32,3.50
1,boleto,19784,2869361.27,145.03,1.00
2,voucher,3866,379436.87,65.70,1.00
3,debit_card,1528,217989.79,142.57,1.00
4,not_defined,3,0.00,0.00,1.00


In [13]:
run_query("""
    SELECT
        c.customer_state                        AS state,
        COUNT(DISTINCT o.order_id)              AS total_orders,
        COUNT(DISTINCT c.customer_unique_id)    AS unique_customers,
        ROUND(SUM(p.payment_value), 2)          AS total_revenue,
        ROUND(AVG(p.payment_value), 2)          AS avg_order_value
    FROM orders o
    JOIN customers c      ON o.customer_id = c.customer_id
    JOIN order_payments p ON o.order_id    = p.order_id
    WHERE o.order_status = 'delivered'
    GROUP BY state
    ORDER BY total_orders DESC
    LIMIT 10
""", "Q8: Top 10 States by Orders and Revenue")


  Q8: Top 10 States by Orders and Revenue


,state,total_orders,unique_customers,total_revenue,avg_order_value
0,SP,40500,39155,5770266.19,136.39
1,RJ,12350,11917,2055690.45,158.08
2,MG,11354,11001,1819277.61,154.12
3,RS,5345,5168,861802.40,155.45
4,PR,4923,4769,781919.55,152.45
5,SC,3546,3449,595208.40,162.58
6,BA,3256,3158,591270.60,169.76
7,DF,2080,2019,346146.17,161.60
8,ES,1995,1928,317682.65,153.62
9,GO,1957,1895,334294.22,163.31


,state,total_orders,unique_customers,total_revenue,avg_order_value
0,SP,40500,39155,5770266.19,136.39
1,RJ,12350,11917,2055690.45,158.08
2,MG,11354,11001,1819277.61,154.12
3,RS,5345,5168,861802.40,155.45
4,PR,4923,4769,781919.55,152.45
5,SC,3546,3449,595208.40,162.58
6,BA,3256,3158,591270.60,169.76
7,DF,2080,2019,346146.17,161.60
8,ES,1995,1928,317682.65,153.62
9,GO,1957,1895,334294.22,163.31


In [14]:
run_query("""
    SELECT
        t.product_category_name_english     AS category,
        ROUND(AVG(r.review_score), 2)       AS avg_review_score,
        COUNT(r.review_id)                  AS total_reviews
    FROM order_reviews r
    JOIN order_items oi  ON r.order_id           = oi.order_id
    JOIN products p      ON oi.product_id        = p.product_id
    JOIN product_category_translation t
                         ON p.product_category_name = t.product_category_name
    GROUP BY category
    HAVING total_reviews >= 100
    ORDER BY avg_review_score DESC
    LIMIT 10
""", "Q9: Top Categories by Review Score (min 100 reviews)")


  Q9: Top Categories by Review Score (min 100 reviews)


,category,avg_review_score,total_reviews
0,books_general_interest,4.45,549
1,books_technical,4.37,266
2,luggage_accessories,4.32,1088
3,food_drink,4.32,279
4,fashion_shoes,4.23,261
5,food,4.22,495
6,stationery,4.19,2507
7,pet_shop,4.19,1939
8,home_appliances,4.17,806
9,computers,4.17,200


,category,avg_review_score,total_reviews
0,books_general_interest,4.45,549
1,books_technical,4.37,266
2,luggage_accessories,4.32,1088
3,food_drink,4.32,279
4,fashion_shoes,4.23,261
5,food,4.22,495
6,stationery,4.19,2507
7,pet_shop,4.19,1939
8,home_appliances,4.17,806
9,computers,4.17,200


In [15]:
run_query("""
    WITH customer_order_counts AS (
        SELECT
            c.customer_unique_id,
            COUNT(DISTINCT o.order_id) AS order_count
        FROM orders o
        JOIN customers c ON o.customer_id = c.customer_id
        GROUP BY c.customer_unique_id
    )
    SELECT
        SUM(CASE WHEN order_count = 1 THEN 1 ELSE 0 END)  AS one_time_buyers,
        SUM(CASE WHEN order_count > 1 THEN 1 ELSE 0 END)  AS repeat_buyers,
        COUNT(*)                                           AS total_customers,
        ROUND(100.0 * SUM(CASE WHEN order_count > 1 THEN 1 ELSE 0 END)
              / COUNT(*), 1)                               AS repeat_rate_pct
    FROM customer_order_counts
""", "Q10: Customer Repeat Purchase Rate")


  Q10: Customer Repeat Purchase Rate


,one_time_buyers,repeat_buyers,total_customers,repeat_rate_pct
0,93099,2997,96096,3.10


,one_time_buyers,repeat_buyers,total_customers,repeat_rate_pct
0,93099,2997,96096,3.10


In [16]:
queries = {
    'q1_business_overview': """
        SELECT COUNT(DISTINCT o.order_id) AS total_orders,
               COUNT(DISTINCT c.customer_unique_id) AS total_customers,
               ROUND(SUM(p.payment_value),2) AS total_revenue,
               ROUND(AVG(p.payment_value),2) AS avg_order_value
        FROM orders o
        JOIN customers c ON o.customer_id = c.customer_id
        JOIN order_payments p ON o.order_id = p.order_id
        WHERE o.order_status = 'delivered'
    """,
    'q2_monthly_revenue': """
        SELECT SUBSTR(o.order_purchase_timestamp,1,7) AS year_month,
               COUNT(DISTINCT o.order_id) AS total_orders,
               ROUND(SUM(p.payment_value),2) AS monthly_revenue
        FROM orders o JOIN order_payments p ON o.order_id = p.order_id
        WHERE o.order_status = 'delivered'
        GROUP BY year_month ORDER BY year_month
    """,
    'q3_top_categories': """
        SELECT t.product_category_name_english AS category,
               COUNT(DISTINCT oi.order_id) AS total_orders,
               ROUND(SUM(oi.price),2) AS total_revenue
        FROM order_items oi
        JOIN products p ON oi.product_id = p.product_id
        JOIN product_category_translation t ON p.product_category_name = t.product_category_name
        GROUP BY category ORDER BY total_revenue DESC LIMIT 10
    """
}

os.makedirs('../outputs/sql_results', exist_ok=True)

for name, sql in queries.items():
    df = pd.read_sql(sql, conn)
    df.to_csv(f'../outputs/sql_results/{name}.csv', index=False)
    print(f'  ✓  Saved {name}.csv')

print('\nQuery results saved to outputs/sql_results/')

  ✓  Saved q1_business_overview.csv
  ✓  Saved q2_monthly_revenue.csv
  ✓  Saved q3_top_categories.csv

Query results saved to outputs/sql_results/


In [17]:
conn.close()
print('Database connection closed ✓')
print('\n✅ Week 2 Complete!')
print('📁 Database saved at: data/olist.db')
print('📁 Query results saved at: outputs/sql_results/')
print('📅 Next: Week 3 — Power BI Dashboard')

Database connection closed ✓

✅ Week 2 Complete!
📁 Database saved at: data/olist.db
📁 Query results saved at: outputs/sql_results/
📅 Next: Week 3 — Power BI Dashboard
